In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pickle
import sys
import itertools

esmDMS_DIR = "/Users/dylanwells/popDMS/esmDMS"

sys.path.insert(0, esmDMS_DIR)
from esmdmsfunctions import (
    run_simulation,
    generate_selection, gaussian_selection, zero_selection,
    load_final_df, get_simulation_results
)
from scipy.stats import pearsonr

In [2]:
import os

# ── Paths
comb_path = esmDMS_DIR + "/data/bg_bf_comb_data"
bg_path   = esmDMS_DIR + "/data/inference_data/BG505"
bf_path   = esmDMS_DIR + "/data/inference_results"

PATHS = {
    "comb":  comb_path,
    "BG505": bg_path,
    "BF520": bf_path,
}

# ── Which layers to run (a representative sample to keep runtime short)
SAMPLE_LAYERS = [0, 5, 10, 15, 20, 25, 30]

# ── Simulation parameters
N_GENS     = 60    # run long so we can see the full fitness trajectory
SAVE_EVERY = 1
FITNESS_FN = 'exp'

# ── Selection functions to compare
SEL_FUNCS = {
    'gaussian': gaussian_selection,
    'single':   generate_selection,
    'neutral':  zero_selection,
}

# ── Parameter grid to sweep for plateau detection
PLATEAU_WINDOWS = [1, 2, 3, 5, 8]
PLATEAU_RTOLS   = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

In [3]:
import os

# ── Paths to analyse (name → directory containing layer0/, layer1/, … subdirs)
comb_path = esmDMS_DIR + "/data/bg_bf_comb_data"
bg_path   = esmDMS_DIR + "/data/inference_data/BG505"
bf_path   = esmDMS_DIR + "/data/inference_results"

PATHS = {
    "comb":  comb_path,
    "BG505": bg_path,
    "BF520": bf_path,
}

# ── Layers to analyze
LAYERS = range(31)

# ── Simulation parameters
N_GENS      = 1000      # number of generations to simulate
SAVE_EVERY  = 1       # save counts every N generations
FITNESS_FN  = 'exp'   # 'plus1' or 'exp'

# ── Eigenvector / PCA parameters
VARIANCE_EXPLAINED_CUTOFF = 0.8   # keep PCs that together explain this fraction of variance
WEIGHT_BY_INITIAL_FREQ    = False  # weight covariance matrix by pre-selection counts

# ── Selection function (generate_selection, gaussian_selection, or zero_selection)
SEL_FUNC = gaussian_selection

# ── Inference flags
RUN_INFERENCE    = False
RUN_GAMMA        = False
CALC_ERROR_BARS  = False
INFER_IGNORED    = False

In [ ]:

# Full covariance matrix method seems to be the best

all_results_fullcov = {}

for path_name, path in PATHS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {path_name}  |  get_simulation_results, method=fullcov")
    print('='*60)

    res = get_simulation_results(
        n_gens=N_GENS,
        embedding_df_path=path,
        sel_func=SEL_FUNC,
        inference=RUN_INFERENCE,
        gamma_analysis=RUN_GAMMA,
        fitness=FITNESS_FN,
        save_every=SAVE_EVERY,
        layers=LAYERS,
        calc_error_bars=True,
        infer_ignored_dims=INFER_IGNORED,
        method="fullcov",
    )
    all_results_fullcov[path_name] = res  
print("\nDone. Datasets completed:", list(all_results_fullcov.keys()))


Dataset: comb  |  get_simulation_results, method=fullcov
Running layer 0...
  Layer 0: running simulation...
  Layer 0: simulation ran for 1000 saved generations.
Running layer 1...
  Layer 1: running simulation...
  Layer 1: simulation ran for 1000 saved generations.
Running layer 2...
  Layer 2: running simulation...
  Layer 2: simulation ran for 1000 saved generations.
Running layer 3...
  Layer 3: running simulation...
  Layer 3: simulation ran for 1000 saved generations.
Running layer 4...
  Layer 4: running simulation...
  Layer 4: simulation ran for 1000 saved generations.
Running layer 5...
  Layer 5: running simulation...
  Layer 5: simulation ran for 1000 saved generations.
Running layer 6...
  Layer 6: running simulation...
  Layer 6: simulation ran for 1000 saved generations.
Running layer 7...
  Layer 7: running simulation...
  Layer 7: simulation ran for 1000 saved generations.
Running layer 8...
  Layer 8: running simulation...
  Layer 8: simulation ran for 1000 saved g

In [ ]:
N_COLS = 6  # subplots per row

for path_name, res in all_results_fullcov.items():
    layers = sorted(res[4].keys())
    n_layers = len(layers)
    n_rows = int(np.ceil(n_layers / N_COLS))

    fig, axes = plt.subplots(n_rows, N_COLS,
                             figsize=(5 * N_COLS, 3.5 * n_rows),
                             constrained_layout=True)
    axes = np.array(axes).flatten()

    for ax_idx, layer in enumerate(layers):
        ax = axes[ax_idx]
        generation_counts = res[4][layer]
        layer_fits = res[0][layer]
        n_reps = len(generation_counts[0])

        for rep in range(n_reps):
            gen_counts_rep = [gen_counts[rep] for gen_counts in generation_counts]
            rep_mean_fits = []
            for gen_counts in gen_counts_rep:
                total_count = np.sum(gen_counts)
                if total_count > 0:
                    freqs = gen_counts / total_count
                    rep_mean_fits.append(np.sum(freqs * layer_fits))
                else:
                    rep_mean_fits.append(0.0)
            ax.plot(rep_mean_fits, lw=1.2, label=f"Rep {rep+1}")

        ax.set_title(f"Layer {layer}", fontsize=9)
        ax.set_xlabel("Generation", fontsize=8)
        ax.set_ylabel("Mean Fitness", fontsize=8)
        ax.tick_params(labelsize=7)
        if ax_idx == 0:
            ax.legend(fontsize=7)

    for ax in axes[n_layers:]:
        ax.set_visible(False)

    fig.suptitle(f"Mean Fitness Trajectories — {path_name}", fontsize=13)
    plt.show()

In [3]:
def mean_fitness_trajectory(generation_counts, fitnesses):
    """Compute the mean fitness averaged over replicates at each saved generation.

    Parameters
    ----------
    generation_counts : list of list-of-arrays
        As returned by run_simulation.  Outer index = saved generation (0 = initial),
        inner index = replicate, each element is a count array over variants.
    fitnesses : ndarray, shape (n_variants,)
        Per-variant fitness values.

    Returns
    -------
    mean_w : ndarray, shape (n_saved_gens,)
        Population-mean fitness at each saved generation, averaged over replicates.
    """
    mean_w = []
    for gen_data in generation_counts:
        rep_means = []
        for rep_counts in gen_data:
            counts = np.asarray(rep_counts, dtype=float)
            total  = counts.sum()
            if total > 0:
                rep_means.append(np.dot(counts, fitnesses) / total)
        mean_w.append(np.mean(rep_means) if rep_means else np.nan)
    return np.array(mean_w)


def find_plateau_cutoff(mean_w, window, rtol):
    """Return the saved-generation index at which the plateau criterion fires.

    Mirrors the logic in run_simulation exactly.  Returns None if the
    criterion never fires over the full trajectory.
    """
    streak = 0
    for i in range(1, len(mean_w)):
        rel_change = abs(mean_w[i] - mean_w[i-1]) / max(abs(mean_w[i-1]), 1e-12)
        if rel_change < rtol:
            streak += 1
            if streak >= window:
                return i   # index into mean_w / generation_counts
        else:
            streak = 0
    return None


print("Helper functions defined.")

Helper functions defined.


In [ ]:
# ── Per-layer s_joint inferred fitness vs true fitness
# One figure per dataset; all layers tiled in a NCOLS-column grid.

for dataset in all_results_fullcov:
    sim_data         = all_results_fullcov[dataset]
    all_layer_fits   = sim_data[0]
    detailed_results = sim_data[2]

    layers_available = sorted(detailed_results.keys())
    n_layers = len(layers_available)
    nrows    = (n_layers + NCOLS - 1) // NCOLS

    fig, axes = plt.subplots(nrows, NCOLS,
                              figsize=(4.5 * NCOLS, 4 * nrows),
                              squeeze=False)
    fig.suptitle(
        f's_joint Inferred vs True Fitness — Dataset: {dataset}',
        fontsize=14, y=1.01
    )

    for idx, layer in enumerate(layers_available):
        row, col = divmod(idx, NCOLS)
        ax = axes[row, col]

        true_z  = z_normalize(np.array(all_layer_fits[layer]))
        s_joint = detailed_results[layer][1]             # (D,)
        emb     = get_embeddings(dataset, layer)         # (N, D)
        inf_z   = z_normalize(inferred_fitness(emb, s_joint, FITNESS_FN))

        _scatter_panel(ax, true_z, inf_z, title=f'Layer {layer}', color='seagreen')

    for idx in range(n_layers, nrows * NCOLS):
        row, col = divmod(idx, NCOLS)
        axes[row, col].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'fitness_scatter_sjoint_{dataset}.png', dpi=80, bbox_inches='tight')
    plt.show()

## Run full simulations (no early stopping)

We run `plateau_window=0` so every simulation goes the full `N_GENS` generations.  
The returned `generation_counts` and `fitnesses` are enough to reconstruct the mean fitness $\bar{w}(t)$ at every saved generation and to replay the plateau-detection logic offline for any parameter combination.

In [4]:
# sim_data[dataset][sel_name][layer] = {'mean_w': ndarray, 'fitnesses': ndarray,
#                                        'n_gens_run': int}
sim_data = {}

SEL_FUNCS = {
    'gaussian': gaussian_selection,
    #'single':   generate_selection,
    #'neutral':  zero_selection,
}



for path_name, path in PATHS.items():
    sim_data[path_name] = {}
    for sel_name, sel_func in SEL_FUNCS.items():
        sim_data[path_name][sel_name] = {}
        for layer in SAMPLE_LAYERS:
            try:
                df_sel = load_final_df(layer, path)
            except Exception as e:
                print(f"  [{path_name}] layer {layer}: could not load ({e})")
                continue

            n_reps = len([c for c in df_sel.columns if 'PreNums' in c])
            initial_counts = [df_sel[f'Rep{r+1}_PreNums'].values for r in range(n_reps)]
            embeddings_len = df_sel['Embedding'].iloc[0].shape[0]
            sel_coeffs = sel_func(embeddings_len)

            gen_counts, fitnesses = run_simulation(
                df_sel, sel_coeffs, initial_counts,
                n_gens=N_GENS, save_every=SAVE_EVERY,
                fitness=FITNESS_FN,
                plateau_window=0,   # no early stopping — full trajectory
            )

            mean_w = mean_fitness_trajectory(gen_counts, fitnesses)
            sim_data[path_name][sel_name][layer] = {
                'mean_w':     mean_w,
                'fitnesses':  fitnesses,
                'n_gens_run': len(mean_w) - 1,
            }
            print(f"  [{path_name}] {sel_name:8s} layer {layer:2d}: "
                  f"{len(mean_w)-1} gens, w̄ {mean_w[0]:.4f} → {mean_w[-1]:.4f}")

print("\nDone.")

  [comb] gaussian layer  0: 60 gens, w̄ 1.3537 → 2.7746
  [comb] gaussian layer  5: 60 gens, w̄ 1.3228 → 7.3168
  [comb] gaussian layer 10: 60 gens, w̄ 1.1212 → 5.6662
  [comb] gaussian layer 15: 60 gens, w̄ 1.0632 → 11.3100
  [comb] gaussian layer 20: 60 gens, w̄ 1.2466 → 1244.3977
  [comb] gaussian layer 25: 60 gens, w̄ 1.1262 → 6.5354
  [comb] gaussian layer 30: 60 gens, w̄ 1.3583 → 7.0961
  [comb] single   layer  0: 60 gens, w̄ 1.0000 → 1.0003
  [comb] single   layer  5: 60 gens, w̄ 0.9996 → 1.0023
  [comb] single   layer 10: 60 gens, w̄ 1.0000 → 1.0020
  [comb] single   layer 15: 60 gens, w̄ 1.0003 → 1.0186
  [comb] single   layer 20: 60 gens, w̄ 0.9999 → 1.0012
  [comb] single   layer 25: 60 gens, w̄ 1.0003 → 1.0611
  [comb] single   layer 30: 60 gens, w̄ 1.0003 → 1.0008
  [comb] neutral  layer  0: 60 gens, w̄ 1.0000 → 1.0000
  [comb] neutral  layer  5: 60 gens, w̄ 1.0000 → 1.0000
  [comb] neutral  layer 10: 60 gens, w̄ 1.0000 → 1.0000
  [comb] neutral  layer 15: 60 gens, w̄ 1.00